In [17]:
import pandas as pd
df = pd.read_csv(r"opentargets_t2d_output\opentargets_t2d_associated_targets_all.csv")
df.head()

,rank_by_score,disease_id,evidence_mode,target_id,target_symbol,target_name,association_score,ds_clinical_precedence,ds_gwas_credible_sets,ds_eva,...,prio_tissueDistribution,prio_tissueSpecificity,prio_geneEssentiality,ds_impc,ds_gene_burden,ds_uniprot_variants,ds_uniprot_literature,prio_isCancerDriverGene,ds_expression_atlas,prio_hasTEP
0,1,MONDO_0005148,direct_only,ENSG00000187486,KCNJ11,potassium inwardly rectifying channel subfamil...,0.865142,0.998500,0.940931,0.914573,...,0.0,0.50,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,MONDO_0005148,direct_only,ENSG00000006071,ABCC8,ATP binding cassette subfamily C member 8,0.864850,0.998500,0.908190,0.940676,...,0.0,0.75,0.0,0.604792,NaN,NaN,NaN,NaN,NaN,NaN
2,3,MONDO_0005148,direct_only,ENSG00000106633,GCK,glucokinase,0.861234,0.609747,0.736958,0.903364,...,0.0,0.50,0.0,0.740716,0.990233,0.607931,0.607931,NaN,NaN,NaN
3,4,MONDO_0005148,direct_only,ENSG00000132170,PPARG,peroxisome proliferator activated receptor gamma,0.848609,0.997083,0.884298,0.847504,...,0.0,0.50,0.0,0.573665,0.261788,NaN,NaN,255.0,NaN,NaN
4,5,MONDO_0005148,direct_only,ENSG00000171105,INSR,insulin receptor,0.788737,0.997991,0.736325,0.616037,...,-1.0,-1.00,0.0,0.551506,NaN,0.413731,0.303965,NaN,NaN,NaN


In [ ]:
df = (
    df.sort_values(
        by=["target_id", "association_score"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset=["target_id"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Before clean:", df.shape)
print("After clean:", df.shape)
print("Unique target IDs:", df["target_id"].nunique())

Before clean: (9957, 35)
After clean: (9687, 35)
Unique target IDs: 9687


In [ ]:
df = (
    df
    .sort_values(
        by=["association_score", "target_symbol"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

df["rank_by_score_clean"] = range(1, len(df) + 1)

In [ ]:
df.to_csv(
    "opentargets_t2d_output/opentargets_t2d_associated_targets_clean_by_target_id.csv",
    index=False,
    encoding="utf-8-sig"
)

In [21]:
df = pd.read_csv("opentargets_t2d_output/opentargets_t2d_associated_targets_clean_by_target_id.csv")
df.head()

,rank_by_score,disease_id,evidence_mode,target_id,target_symbol,target_name,association_score,ds_clinical_precedence,ds_gwas_credible_sets,ds_eva,...,prio_tissueSpecificity,prio_geneEssentiality,ds_impc,ds_gene_burden,ds_uniprot_variants,ds_uniprot_literature,prio_isCancerDriverGene,ds_expression_atlas,prio_hasTEP,rank_by_score_clean
0,1,MONDO_0005148,direct_only,ENSG00000187486,KCNJ11,potassium inwardly rectifying channel subfamil...,0.865142,0.998500,0.940931,0.914573,...,0.50,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2,MONDO_0005148,direct_only,ENSG00000006071,ABCC8,ATP binding cassette subfamily C member 8,0.864850,0.998500,0.908190,0.940676,...,0.75,0.0,0.604792,NaN,NaN,NaN,NaN,NaN,NaN,2
2,3,MONDO_0005148,direct_only,ENSG00000106633,GCK,glucokinase,0.861234,0.609747,0.736958,0.903364,...,0.50,0.0,0.740716,0.990233,0.607931,0.607931,NaN,NaN,NaN,3
3,4,MONDO_0005148,direct_only,ENSG00000132170,PPARG,peroxisome proliferator activated receptor gamma,0.848609,0.997083,0.884298,0.847504,...,0.50,0.0,0.573665,0.261788,NaN,NaN,255.0,NaN,NaN,4
4,5,MONDO_0005148,direct_only,ENSG00000171105,INSR,insulin receptor,0.788737,0.997991,0.736325,0.616037,...,-1.00,0.0,0.551506,NaN,0.413731,0.303965,NaN,NaN,NaN,5


In [22]:
print("Before dedup:", df.shape)

df_dedup = df.drop_duplicates()

print("After dedup:", df_dedup.shape)

target_id_counts = df_dedup["target_id"].value_counts()
print(target_id_counts[target_id_counts > 1].head(30))

Before dedup: (9687, 36)
After dedup: (9687, 36)
Series([], Name: count, dtype: int64)


In [23]:
thresholds = [0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

score_threshold_summary = pd.DataFrame({
    "score_threshold": thresholds,
    "n_targets_score_ge_threshold": [
        (df["association_score"] >= t).sum()
        for t in thresholds
    ]
})

score_threshold_summary

,score_threshold,n_targets_score_ge_threshold
0,0.00,9687
1,0.01,6268
2,0.05,3210
3,0.10,2123
4,0.20,1479
5,0.30,911
6,0.40,472
7,0.50,189
8,0.60,77


In [24]:
bins = [0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.4, 0.6, 1.0]

df["score_bin"] = pd.cut(
    df["association_score"],
    bins=bins,
    include_lowest=True
)

df["score_bin"].value_counts().sort_index()

score_bin
(-0.001, 0.001]       5
(0.001, 0.01]      3414
(0.01, 0.05]       3058
(0.05, 0.1]        1087
(0.1, 0.2]          644
(0.2, 0.4]         1007
(0.4, 0.6]          395
(0.6, 1.0]           77
Name: count, dtype: int64

In [25]:
summary_by_threshold = []

for threshold in [0.05, 0.1, 0.2, 0.3, 0.5]:
    subset = df[df["association_score"] >= threshold]
    
    summary_by_threshold.append({
        "threshold": threshold,
        "n_targets": len(subset),
        "with_gwas": subset["ds_gwas_credible_sets"].notna().sum(),
        "with_clinical": subset["ds_clinical_precedence"].notna().sum(),
        "with_europepmc": subset["ds_europepmc"].notna().sum(),
        "with_gene_burden": subset["ds_gene_burden"].notna().sum(),
        "with_eva": subset["ds_eva"].notna().sum(),
    })

pd.DataFrame(summary_by_threshold)

,threshold,n_targets,with_gwas,with_clinical,with_europepmc,with_gene_burden,with_eva
0,0.05,3210,2421,193,2160,16,43
1,0.10,2123,1920,143,1376,15,41
2,0.20,1479,1383,128,993,10,38
3,0.30,911,832,109,679,10,36
4,0.50,189,132,77,173,9,26


In [29]:
df.shape

(9687, 37)

In [30]:
disgenet_top30 = [
    "ACE", "DPP4", "GCK", "GIP", "IAPP", "APOB", "IGF1", "INS",
    "PPARG", "CDKAL1", "RETN", "THADA", "SLC2A2", "WFS1", "KL",
    "HNF1A", "TNF", "FTO", "LEPR", "PAX4", "GHRL", "SLC30A8",
    "JAZF1", "PPP1R3A", "PTPN1", "IDE", "INSR", "IGF2BP2",
    "SIRT1", "GLP1R"
]

disgenet_in_ot = df[
    df["target_symbol"].isin(disgenet_top30)
][
    ["target_symbol", "association_score", "ds_gwas_credible_sets", "ds_clinical_precedence", "ds_europepmc"]
].sort_values("association_score", ascending=False)

disgenet_in_ot

,target_symbol,association_score,ds_gwas_credible_sets,ds_clinical_precedence,ds_europepmc
2,GCK,0.861234,0.736958,0.609747,0.914113
3,PPARG,0.848609,0.884298,0.997083,0.901977
4,INSR,0.788737,0.736325,0.997991,0.390372
6,HNF1A,0.779641,0.885408,NaN,0.954945
8,WFS1,0.769511,0.916338,NaN,0.522792
9,GLP1R,0.766700,0.898974,0.997767,0.989240
10,SLC2A2,0.742146,0.835853,NaN,0.561185
13,SLC30A8,0.697409,0.942137,NaN,0.753412
15,PAX4,0.676198,0.890530,NaN,0.382821
21,DPP4,0.643283,NaN,0.998195,0.954189


In [31]:
disgenet_overlap_by_threshold = pd.DataFrame({
    "threshold": [0.05, 0.10, 0.20, 0.30, 0.50],
    "disgenet_top30_recovered": [
        (
            (df["association_score"] >= t)
            & (df["target_symbol"].isin(disgenet_top30))
        ).sum()
        for t in [0.05, 0.10, 0.20, 0.30, 0.50]
    ]
})

disgenet_overlap_by_threshold

,threshold,disgenet_top30_recovered
0,0.05,30
1,0.10,29
2,0.20,22
3,0.30,21
4,0.50,19
